AI VISION LAB · 05 / STUDENT WORKBOOK

# Mission 02. 회전 증강 이미지 관찰하기

이미지 데이터 증강과 재현 가능한 실험

**학습 목표**

- `train_ds`에서 이미지 한 장과 정답 라벨을 가져올 수 있다.
- 같은 이미지에 무작위 회전을 여러 번 적용하여 결과를 시각화할 수 있다.
- `take()`, 배치 차원, `training=True`의 역할을 설명할 수 있다.

### VS Code 실행 준비

`DATA_ROOT`를 4강 웨이퍼 이미지 `Train` 폴더가 있는 위치로 설정하세요.

커널 목록에서 TensorFlow가 설치된 Python 환경을 선택하고 위에서부터 실행하세요.

In [ ]:
from pathlib import Path
from tensorflow import keras
from tensorflow.keras import layers

COURSE_DIR = Path('D:\\2604340036 - 송성민\\2학기\\반도체장비AI비전실무\\2026-AIVISION_05강')
DATA_ROOT = COURSE_DIR.parent / "wafermap_image_data_261021"

IMG_SIZE = 64
class_names = ["Center", "Edge-Loc", "Edge-Ring", "Loc", "Scratch"]
BATCH_SIZE = 32
TRAIN_DIR = DATA_ROOT / "Train"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}

def check_image_folders(directory):
    missing = [
        name for name in class_names
        if not any(
            path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
            for path in (directory / name).rglob("*")
        )
    ]
    if missing:
        raise FileNotFoundError(
            f"{directory}에 이미지가 없는 클래스: {', '.join(missing)}. "
            "첫 셀의 DATA_ROOT를 4강 Train/Test가 있는 폴더로 설정하세요."
        )

In [ ]:
import numpy as np
import tensorflow as tf

check_image_folders(TRAIN_DIR)

# Python에서 파일을 읽어 Windows의 한글 경로를 처리합니다.
image_paths = []
image_labels = []
for label, class_name in enumerate(class_names):
    class_paths = sorted(
        path for path in (TRAIN_DIR / class_name).rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )
    image_paths.extend(class_paths)
    image_labels.extend([label] * len(class_paths))

# 시드 42로 같은 이미지 목록을 학습 80%, 검증 20%로 나눕니다.
seed = 42
indices = np.random.RandomState(seed).permutation(len(image_paths))
val_count = int(len(indices) * 0.2)
train_indices = indices[:-val_count]
val_indices = indices[-val_count:]

def decode_image(image_bytes, label):
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    return image, label

def make_dataset(selected_indices, shuffle=False):
    image_bytes = [image_paths[i].read_bytes() for i in selected_indices]
    labels = np.asarray([image_labels[i] for i in selected_indices], dtype=np.int32)
    dataset = tf.data.Dataset.from_tensor_slices((image_bytes, labels))
    if shuffle:
        dataset = dataset.shuffle(BATCH_SIZE * 8, seed=seed)
    dataset = dataset.map(decode_image, num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_indices, shuffle=True)
val_ds = make_dataset(val_indices)
print(f"학습 이미지: {len(train_indices):,}장 / 검증 이미지: {len(val_indices):,}장")

In [ ]:
# 회전 증강 레이어 생성
data_augmentation = keras.Sequential([
    layers.RandomRotation(
        factor=0.25,
        fill_mode="constant",
        fill_value=0.0,
        interpolation="nearest"
    )
])

## 1. 실습 준비: 데이터셋과 증강 레이어

Mission 01에서 만든 `data_augmentation`을 사용하여 실제 웨이퍼맵을 회전시켜 보자. 이번 미션에서는 **모델을 학습하지 않고 이미지 변형 결과만 확인한다.**

**필요한 준비물**

1. 4강에서 생성한 `train_ds`와 `class_names`
2. Mission 01에서 생성한 `data_augmentation` (±90° 회전, 검은 배경, nearest 보간)
3. `matplotlib.pyplot` (이미지 출력)

Colab 런타임을 새로 시작했다면 데이터셋과 증강 레이어를 만드는 셀부터 실행해야 한다.

## 2. 데이터셋에서 이미지 한 장 가져오기
`train_ds`는 이미지를 한 장씩이 아니라 **배치(batch)** 단위로 제공한다. 4강에서 `BATCH_SIZE=32`로 설정했으므로 일반적으로 한 배치에는 이미지 32장과 대응하는 라벨 32개가 들어 있다.

In [ ]:
# 앞에서 배치 하나 선택
for images, labels in train_ds.take(1):
    print("이미지 배치:", images.shape)
    print("라벨 배치:", labels.shape)

`take(n)`은 데이터셋의 앞에서 **n개 요소**를 가져오는 메서드이다. 여기서는 요소 하나가 이미지 한 장이 아니라 배치 하나이므로, `take(1)`을 순회하는 `for`문은 한 번 실행된다.
| 코드 | 선택되는 데이터 | 반복문 실행 |
| --- | --- | --- |
| `take(0)` | 없음 | 0회 |
| `take(1)` | 배치 1개 | 1회 |
| `take(3)` | 배치 최대 3개 | 최대 3회 |
이제 가져온 배치에서 **첫 번째 이미지와 라벨**을 선택하자.

In [ ]:
for images, labels in train_ds.take(1):
    # 배치에서 이미지 한 장만 선택
    sample_image = images[0]
    sample_label = labels[0]

print("이미지 형태:", sample_image.shape)
print("정답 클래스:", class_names[int(sample_label)])

`sample_image.shape`은 `(64, 64, 3)`으로 출력될 것이다. 높이 64, 너비 64, RGB 채널 3개를 뜻한다.

`take(1)`의 `1`은 가져올 배치의 개수이고, `images[0]`의 `0`은 배치 안에서 첫 번째 이미지를 가리키는 인덱스이다. 두 숫자의 의미를 구분하자.

## 3. 이미지에 회전 증강 적용하기
`data_augmentation`은 `keras.Sequential`로 만든 객체이다. 함수처럼 호출하면 입력 이미지를 내부 레이어에 순서대로 통과시켜 변형 결과를 반환한다.

In [ ]:
augmented = data_augmentation(
    sample_image[None, ...],
    training=True
)

print("변형 결과 형태:", augmented.shape)

| 표현 | 의미 |
| --- | --- |
| `sample_image[None, ...]` | 이미지 한 장에 배치 차원을 추가: `(64,64,3)` → `(1,64,64,3)` |
| `data_augmentation(...)` | Sequential 객체를 호출해 포함된 `RandomRotation` 실행 |
| `training=True` | 훈련 모드 동작을 활성화하여 무작위 회전 적용 |

**훈련 모드(training mode)**는 학습 과정에서만 다르게 작동하는 레이어의 동작을 지정한다. `RandomRotation`은 훈련 모드에서 이미지를 회전시키고, 추론 모드에서는 회전시키지 않는다.

`training=True`는 **모델을 학습시키라는 명령이 아니다.** 실제 가중치 학습은 `fit()`에서 이루어진다. 현재 증강 레이어에는 학습할 가중치가 없다.

## 4. 원본과 회전 이미지 6장 출력하기
같은 이미지라도 증강 레이어를 여러 번 호출하면 무작위 회전 각도가 달라질 수 있다. 다음 코드를 작성하여 원본과 증강 이미지를 나란히 비교해 보자.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 4))

# 원본 이미지
plt.subplot(1, 7, 1)
plt.imshow(sample_image.numpy().astype("uint8"))
plt.title("Original")
plt.axis("off")

# 동일한 이미지를 6번 무작위 회전
for i in range(6):
    augmented = data_augmentation(
        sample_image[None, ...], training=True
    )

    plt.subplot(1, 7, i + 2)
    plt.imshow(augmented[0].numpy().astype("uint8"))
    plt.title(f"Rotation {i + 1}")
    plt.axis("off")

plt.tight_layout()
plt.show()

이 코드는 `subplot(1, 7, ...)`을 사용하여 **1행 7열**에 이미지를 배치한다. 회전 결과의 실제 각도는 실행할 때마다 달라질 수 있다.

표시된 결함 유형이 적합한지 판단하기 어려우면, 다른 이미지로 다시 실행하여 비교해도 된다.

## 5. 실행 결과 관찰 및 코드 이해 활동

출력된 이미지를 확인하고 질문에 답해보자. 관찰 결과는 이미지와 함께 기록해도 좋다.

**Q2.1.**

`take(1)`의 `1`과 `images[0]`의 `0`은 각각 무엇을 의미하는가? `take(2)`로 변경하면 `for`문은 몇 번 실행되는가?

**답변**

`take(1)`의 `1`은 가져올 **배치 수**, `images[0]`의 `0`은 배치 안의 **첫 번째 이미지 인덱스**입니다. 배치가 두 개 이상이면 `take(2)`에서 반복문이 두 번 실행됩니다.

**Q2.2.**

코드에서 `sample_image`의 차원을 `(64, 64, 3)`에서 `(1, 64, 64, 3)`으로 바꾸는 부분을 찾고, 앞에 추가된 `1`이 무엇을 나타내는지 설명하시오.

**답변**

`sample_image[None, ...]`에서 맨 앞에 배치 차원을 추가합니다. `(1, 64, 64, 3)`의 `1`은 배치에 이미지가 한 장 있다는 뜻입니다.

**Q2.3.**

`training=True`를 `training=False`로 바꾸어 다시 실행하시오. 출력된 회전 이미지에 어떤 변화가 있는가? 그 이유를 설명하시오.

**답변**

`training=False`에서는 `RandomRotation`이 입력을 그대로 통과시킵니다. 확인에 사용한 이미지에서 변형 결과와 원본의 픽셀값이 모두 같았습니다.

**Q2.4.**

같은 이미지에 무작위 회전을 적용한 여섯 결과를 관찰하시오. 결함 패턴이 여전히 같은 클래스로 보이는가? 회전 후 분류가 모호해지는 사례가 있다면 어떤 특징 때문인지 작성하시오.

**답변**

검증용 **Loc** 이미지의 여섯 회전 결과에서 불량 덩어리의 방향은 바뀌었지만, 국부적으로 모인 모양은 유지됐습니다. 가장자리 쪽 불량이 화면 경계에서 잘리면 불량의 모양이나 위치를 구분하기 어려워질 수 있습니다.

**Q2.5.**

Mission 01에서 설정한 `interpolation="nearest"`를 `"bilinear"`로 변경하고 같은 이미지의 결과를 비교하시오. 결함 영역의 경계와 픽셀값 표현에서 어떤 차이가 나타나는가?

**답변**

같은 각도로 비교하면 `nearest`는 기존 픽셀값을 가져와 경계가 비교적 선명합니다. `bilinear`는 주변값을 섞어 중간 밝기를 만들므로 경계가 더 부드럽게 보입니다.

In [ ]:
# 같은 원본을 추론 모드와 두 보간 방식으로 비교합니다.
sample_batch = sample_image[None, ...]
unchanged = data_augmentation(sample_batch, training=False)[0]

nearest = layers.RandomRotation(
    0.25, fill_mode="constant", fill_value=0.0,
    interpolation="nearest", seed=42,
)
bilinear = layers.RandomRotation(
    0.25, fill_mode="constant", fill_value=0.0,
    interpolation="bilinear", seed=42,
)
images_to_compare = [
    sample_image, unchanged,
    nearest(sample_batch, training=True)[0],
    bilinear(sample_batch, training=True)[0],
]
titles = ["Original", "Training=False", "Nearest", "Bilinear"]
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, image, title in zip(axes, images_to_compare, titles):
    ax.imshow(image.numpy().astype("uint8"))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()